# ۳. آموزش/سازگارسازی اختیاری SURT

این مرحله برای ساختن اولین خروجی مدل پایه لازم نیست. ابتدا اجرای وزن پایه در مرحلهٔ 12 باید تأیید شود.
دادهٔ train برای تغییر وزن، validation برای انتخاب وزن و تنظیمات، و test فقط برای ارزیابی نهایی است.
رابط آموزش پروژه موجود است اما backend واقعیِ اتصال به recipe رسمی هنوز ارائه نشده است.
`--validate-only` هیچ وزنی تولید نمی‌کند. فعال‌کردن GPU یا نوشتن نام مدل در YAML این کمبود را برطرف نمی‌کند.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/manzourali/English-Streaming-ASR.git"
REPOSITORY_REF = "master"
ROOT = Path("/kaggle/working/English-Streaming-ASR")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(ROOT)], check=True)
os.chdir(ROOT)
if not (ROOT / "scripts/check_surt_assets.py").is_file():
    raise RuntimeError("This GitHub checkout predates the Kaggle workflow update. Publish the updated source, then start a fresh session.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[kaggle,dev]"], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
ARTIFACTS = Path("/kaggle/working/asr_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str, arguments)], check=True)

In [ ]:
import yaml
DATA_ROOT = Path("/kaggle/working/asr_artifacts/data/pilot_v1")
TRAINING_FACTORY = None
INITIAL_CHECKPOINT = None
RESUME_FROM = None
RUN_TRAINING = False
training_override = ARTIFACTS / "surt_training.yaml"
training_values = {
    "dataset": {f"{name}_manifest": str(DATA_ROOT / "manifests" / f"{name}.jsonl")
                for name in ("train", "validation", "test")},
    "model": {"backend_factory": TRAINING_FACTORY, "checkpoint": INITIAL_CHECKPOINT},
    "training": {"epochs": 3, "batch_size": 1, "resume_from": RESUME_FROM},
    "paths": {"outputs": str(ARTIFACTS / "training/results"),
              "checkpoints": str(ARTIFACTS / "training/checkpoints")}}
training_override.write_text(yaml.safe_dump(training_values, sort_keys=False))
run_script("train_asr.py", "--override", training_override, "--validate-only")

## تفاوت دو backend

`model.backend_factory` در تنظیمات inference باید `SURTBackend` برگرداند؛ همین کلید در تنظیمات training باید `ExternalSurtTrainingBackend` برگرداند.
این دو قابل جایگزینی نیستند. آموزش واقعی باید شبکه، tokenizer، محاسبهٔ loss، optimizer، scheduler، precision و ذخیره/بازیابی را از recipe نسخه‌بندی‌شده پیاده کند.
تنظیم batch و epochs بدون آن کافی نیست. در این پروژه LoRA آماده و اثبات‌شده برای SURT نداریم.
recipe رسمی `train.py` و `train_adapt.py` دارد، اما دستور چند GPU مقاله را بدون تطبیق به Kaggle کپی نمی‌کنیم.

In [ ]:
if RUN_TRAINING:
    if not TRAINING_FACTORY or not INITIAL_CHECKPOINT:
        raise RuntimeError("A verified training backend and matching initialization checkpoint are required.")
    run_script("train_asr.py", "--override", training_override)
else:
    print("Validation only. No training ran and no fine-tuned weights were created.")

## استفاده از وزن حاصل

پس از آموزش واقعی باید checkpoint انتخاب‌شده بر اساس validation را ذخیره کنید؛ optimizer و scheduler و state نیز برای ادامهٔ آموزش لازم‌اند.
سپس checkpoint آموزشی باید به native export سازگار تبدیل شود. **اسکریپت این تبدیل هنوز در پروژه نیست**؛ خروجی trainer مستقیماً ورودی loader فعلی نیست.
آموزش و دانلود مدل دو مسیر جایگزین برای به‌دست‌آوردن وزن‌اند؛ export مرحله‌ای جداگانه است.
بعد از ساخت و بررسی export، مرحلهٔ 12 را با `EXPORT_ROOT` جدید و `SYSTEM_NAME="surt_finetuned"` تکرار کنید.
سپس مرحلهٔ 14 نتایج پایه و آموزش‌دیده را روی test یکسان مقایسه می‌کند.